<h1>NYC TLC Yellow Taxi Trip Records: Research project</h1>
<h2>1. Import data</h2>

In [2]:
import pandas as pd

df = pd.read_parquet("../data/yellow_tripdata_2026-01.parquet")

In [4]:
df.head(-5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.20,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.90,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.70,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.70,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.50,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724879,2,2026-01-31 23:19:37,2026-01-31 23:41:09,NaN,3.54,NaN,NaN,87,79,0,20.21,0.00,0.5,0.00,0.0,1.0,24.96,NaN,NaN,0.75
3724880,1,2026-01-31 23:25:55,2026-01-31 23:44:12,NaN,1.50,NaN,NaN,246,170,0,19.40,0.00,0.5,0.00,0.0,1.0,37.94,NaN,NaN,0.75
3724881,2,2026-01-31 23:30:50,2026-01-31 23:58:46,NaN,3.83,NaN,NaN,33,36,0,32.17,0.00,0.5,0.00,0.0,1.0,33.67,NaN,NaN,0.00
3724882,2,2026-01-31 23:11:42,2026-01-31 23:19:54,NaN,0.70,NaN,NaN,114,144,0,27.11,0.00,0.5,0.00,0.0,1.0,31.86,NaN,NaN,0.75


In [31]:
money_cols = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee"
]

df[money_cols].max().sort_values(ascending=False)

total_amount             2560.20
fare_amount              2555.20
tip_amount                766.00
tolls_amount              122.22
Airport_fee                26.75
extra                      17.46
mta_tax                     4.75
congestion_surcharge        2.50
improvement_surcharge       1.00
cbd_congestion_fee          0.75
dtype: float64

In [2]:
print(df.shape)
df.info()
print(df.dtypes)

(3724889, 20)
<class 'pandas.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee        

In [3]:
df.isna().sum().sort_values(ascending=False)

passenger_count          1088058
congestion_surcharge     1088058
store_and_fwd_flag       1088058
RatecodeID               1088058
Airport_fee              1088058
tpep_dropoff_datetime          0
VendorID                       0
tpep_pickup_datetime           0
DOLocationID                   0
payment_type                   0
trip_distance                  0
PULocationID                   0
extra                          0
fare_amount                    0
mta_tax                        0
tip_amount                     0
improvement_surcharge          0
tolls_amount                   0
total_amount                   0
cbd_congestion_fee             0
dtype: int64

In [22]:
cols = [
    'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID',
    'payment_type', 'tip_amount', 'tolls_amount'
]

dup_mask = df[cols].duplicated(keep=False)
dups = df[dup_mask].copy()

# Проверяем: сумма total_amount внутри каждой группы дубликатов ~ 0?
group_sums = dups.groupby(cols)['total_amount'].sum()
print("Сумма total_amount по группам дубликатов:")
print(group_sums.describe())
print(f"\nГрупп, где сумма ~ 0: {((group_sums.abs() < 0.01).sum())}")
print(f"Всего групп: {len(group_sums)}")

# Проверяем payment_type внутри групп
pt_variety = dups.groupby(cols)['payment_type'].nunique()
print(f"\nГрупп с разными payment_type: {(pt_variety > 1).sum()}")

# Смотрим распределение payment_type среди дубликатов
print("\nРаспределение payment_type в дубликатах:")
print(dups['payment_type'].value_counts().sort_index())


Сумма total_amount по группам дубликатов:
count    32485.000000
mean        -0.004393
std          0.560568
min        -75.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.000000
Name: total_amount, dtype: float64

Групп, где сумма ~ 0: 32483
Всего групп: 32485

Групп с разными payment_type: 0

Распределение payment_type в дубликатах:
payment_type
1      388
2    14902
3     7520
4    42160
Name: count, dtype: int64


In [5]:
df.nunique().sort_values()

store_and_fwd_flag             2
congestion_surcharge           3
cbd_congestion_fee             3
VendorID                       4
improvement_surcharge          4
payment_type                   5
mta_tax                        6
RatecodeID                     7
Airport_fee                    8
passenger_count               10
extra                         45
DOLocationID                 260
PULocationID                 262
tolls_amount                1495
tip_amount                  4326
trip_distance               4777
fare_amount                12867
total_amount               21228
tpep_dropoff_datetime    1755995
tpep_pickup_datetime     1757191
dtype: int64

In [7]:
print(f"tpep_pickup_datetime (min):  {df["tpep_pickup_datetime"].min()}")
print(f"tpep_pickup_datetime (max):  {df["tpep_pickup_datetime"].max()}")
print(f"tpep_dropoff_datetime (min): {df["tpep_dropoff_datetime"].min()}")
print(f"tpep_dropoff_datetime (max): {df["tpep_dropoff_datetime"].max()}")

tpep_pickup_datetime (min):  2025-12-31 23:57:29
tpep_pickup_datetime (max):  2026-02-01 00:45:01
tpep_dropoff_datetime (min): 2025-12-31 23:57:32
tpep_dropoff_datetime (max): 2026-02-01 23:35:31


In [8]:
df["trip_distance"].describe()

count    3.724889e+06
mean     6.455647e+00
std      6.488855e+02
min      0.000000e+00
25%      1.000000e+00
50%      1.810000e+00
75%      3.730000e+00
max      2.690975e+05
Name: trip_distance, dtype: float64

In [9]:
df["fare_amount"].describe()

count    3.724889e+06
mean     2.080425e+01
std      1.892701e+01
min     -2.555200e+03
25%      1.000000e+01
50%      1.560000e+01
75%      2.610000e+01
max      2.555200e+03
Name: fare_amount, dtype: float64

In [10]:
df["passenger_count"].value_counts(dropna=False).sort_index()

passenger_count
0.0      14787
1.0    2150994
2.0     334370
3.0      72864
4.0      49738
5.0       9184
6.0       4887
7.0          2
8.0          4
9.0          1
NaN    1088058
Name: count, dtype: int64

In [12]:
print(f"{df[df["passenger_count"].isna()].shape}")
print(f"{df[df["passenger_count"].isna()]["VendorID"].value_counts(dropna=False)}")
print(f"{df[df["passenger_count"].isna()]["payment_type"].value_counts(dropna=False)}")

(1088058, 20)
VendorID
2    944635
1    139406
6      4017
Name: count, dtype: int64
payment_type
0    1088058
Name: count, dtype: int64


In [13]:
df.nlargest(20, "trip_distance")[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "PULocationID",
        "DOLocationID"
    ]
]

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount,PULocationID,DOLocationID
3524923,2026-01-28 01:08:00,2026-01-28 01:16:00,269097.48,12.19,16.94,79,144
3632367,2026-01-30 13:00:00,2026-01-30 13:38:00,237567.19,72.99,81.95,42,130
3285750,2026-01-21 16:31:00,2026-01-21 16:40:00,236216.69,13.58,21.33,234,107
3131676,2026-01-17 10:40:00,2026-01-17 10:57:00,236183.06,22.75,34.96,137,82
2704619,2026-01-02 21:31:00,2026-01-02 21:55:00,221544.91,31.21,39.55,249,41
2688274,2026-01-02 07:07:00,2026-01-02 07:18:00,221436.73,27.61,32.36,231,161
3131406,2026-01-17 10:39:00,2026-01-17 10:56:00,196490.48,33.59,38.34,13,162
3571687,2026-01-29 07:51:00,2026-01-29 08:26:00,192081.68,60.20,64.95,33,246
2820951,2026-01-07 17:51:00,2026-01-07 17:56:00,184908.33,12.18,16.93,164,186
3499856,2026-01-27 14:46:00,2026-01-27 16:10:00,183363.75,71.01,75.76,230,74


In [14]:
(df["trip_distance"] == 0).sum()

np.int64(125738)

In [15]:
df[df["fare_amount"] < 0][
    [
        "fare_amount",
        "total_amount",
        "payment_type",
        "trip_distance",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime"
    ]
].head(20)

,fare_amount,total_amount,payment_type,trip_distance,tpep_pickup_datetime,tpep_dropoff_datetime
74,-5.1,-10.10,4,0.63,2026-01-01 00:54:05,2026-01-01 00:56:36
248,-22.6,-28.35,4,1.92,2026-01-01 00:43:29,2026-01-01 01:09:26
368,-3.0,-8.75,4,0.02,2026-01-01 00:37:39,2026-01-01 00:38:17
428,-3.0,-8.75,4,0.03,2026-01-01 00:01:24,2026-01-01 00:02:02
649,-22.6,-28.35,2,3.12,2026-01-01 00:10:13,2026-01-01 00:35:15
871,-8.6,-13.60,4,1.03,2026-01-01 00:33:23,2026-01-01 00:40:16
980,-70.0,-71.50,3,0.01,2025-12-31 23:57:29,2025-12-31 23:57:32
1046,-3.7,-9.45,3,0.14,2026-01-01 00:01:52,2026-01-01 00:04:22
1076,-17.7,-22.70,2,3.63,2026-01-01 00:24:04,2026-01-01 00:36:44
1106,-3.0,-8.00,4,0.01,2026-01-01 00:30:51,2026-01-01 00:31:00


In [16]:
(df["fare_amount"] < 0).sum()

np.int64(39463)

In [17]:
df["trip_duration_min"] = (
    df["tpep_dropoff_datetime"] -
    df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["trip_duration_min"].describe()

count    3.724889e+06
mean     1.719316e+01
std      2.517802e+01
min     -1.170000e+01
25%      8.066667e+00
50%      1.336667e+01
75%      2.120000e+01
max      7.507900e+03
Name: trip_duration_min, dtype: float64

In [18]:
df.nlargest(20, "trip_duration_min")[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_duration_min",
        "trip_distance",
        "fare_amount"
    ]
]

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_min,trip_distance,fare_amount
1560244,2026-01-19 00:35:57,2026-01-24 05:43:51,7507.900000,0.00,70.00
572461,2026-01-08 12:06:24,2026-01-12 15:20:38,5954.233333,0.00,3.70
710966,2026-01-09 20:41:45,2026-01-13 12:55:05,5293.333333,2.90,15.60
2027357,2026-01-23 22:38:23,2026-01-27 12:32:58,5154.583333,4.86,67.40
100303,2026-01-02 14:11:42,2026-01-05 20:44:12,4712.500000,0.00,25.50
1740426,2026-01-21 09:04:19,2026-01-24 08:50:35,4306.266667,21.73,82.80
469655,2026-01-07 10:56:28,2026-01-10 01:26:30,3750.033333,0.00,40.50
2127024,2026-01-25 00:43:10,2026-01-27 11:28:52,3525.700000,48.65,-2555.20
2127025,2026-01-25 00:43:10,2026-01-27 11:28:52,3525.700000,48.65,2555.20
2050814,2026-01-24 09:28:41,2026-01-26 11:51:47,3023.100000,2.10,10.00


In [19]:
(df["trip_duration_min"] < 0).sum()

np.int64(1)

In [22]:
zero_distance = df[df["trip_distance"] == 0]

zero_distance["fare_amount"].describe()

count    125738.000000
mean         27.965840
std          30.721396
min        -990.000000
25%          13.950000
50%          22.550000
75%          34.900000
max        2500.000000
Name: fare_amount, dtype: float64

In [25]:
zero_distance["payment_type"].value_counts(dropna=False)

payment_type
0    92717
1    18249
2     7643
4     3889
3     3240
Name: count, dtype: int64

In [26]:
zero_distance["trip_duration_min"].describe()

count    125738.000000
mean         15.537019
std          43.075749
min           0.000000
25%           4.516667
50%          13.300000
75%          21.116667
max        7507.900000
Name: trip_duration_min, dtype: float64

In [24]:
zero_distance[
    ["trip_duration_min", "fare_amount", "total_amount", "payment_type"]
].describe()

,trip_duration_min,fare_amount,total_amount,payment_type
count,125738.000000,125738.000000,125738.000000,125738.000000
mean,15.537019,27.965840,34.300725,0.467727
std,43.075749,30.721396,33.532245,0.946729
min,0.000000,-990.000000,-991.000000,0.000000
25%,4.516667,13.950000,18.950000,0.000000
50%,13.300000,22.550000,28.160000,0.000000
75%,21.116667,34.900000,41.410000,1.000000
max,7507.900000,2500.000000,2500.000000,4.000000


In [27]:
negative_fares = df[df["fare_amount"] < 0]

negative_fares["payment_type"].value_counts(dropna=False)

payment_type
4    26039
2     8633
3     4492
1      267
0       32
Name: count, dtype: int64

In [28]:
negative_fares["VendorID"].value_counts(dropna=False)

VendorID
2    39463
Name: count, dtype: int64

In [29]:
negative_fares[
    ["fare_amount", "total_amount", "payment_type", "trip_distance"]
].describe()

,fare_amount,total_amount,payment_type,trip_distance
count,39463.000000,39463.000000,39463.000000,39463.000000
mean,-24.554129,-30.703067,3.425107,4.082969
std,34.864839,36.512232,0.855721,6.123440
min,-2555.200000,-2560.200000,0.000000,0.000000
25%,-31.000000,-36.050000,3.000000,0.630000
50%,-13.500000,-18.650000,4.000000,1.480000
75%,-7.200000,-12.650000,4.000000,5.020000
max,-0.010000,17.940000,4.000000,142.920000


In [30]:
for threshold in [10, 20, 50, 100, 500, 1000]:
    print(
        threshold,
        (df["trip_distance"] > threshold).sum()
    )

10 279512
20 31202
50 620
100 162
500 111
1000 111


In [31]:
for threshold in [60, 120, 180, 360, 720, 1440]:
    print(
        threshold,
        (df["trip_duration_min"] > threshold).sum()
    )

60 69218
120 4523
180 1755
360 1309
720 936
1440 35
